# ML-JET-Ext: Publication moment images

Reproduces the layout of the original 4×3 reference figure, updated to **five $Q_0$ rows × seven $\alpha_s$ columns**. Uses existing normalized `(5,32,32)` tensors; **no moments are recalculated**.

**Coordinate convention:** `process_unfolded_config.py` used `np.histogram2d(phi, eta, range=[(-0.5,0.5),(-0.5,0.5)])`. Consequently we display `moment.T`, producing horizontal $\phi$ and vertical $\eta$ over $[-0.5,0.5]$ for each coordinate.


In [ ]:
from pathlib import Path
import csv
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.cm import ScalarMappable
from IPython.display import display

DATASET_ROOT = Path('/mnt/Research/arsalan/pTHat50-250_eCM5020_c0005_pT90-110_R0p4_agg500_m1-5_normalized')
OUTPUT_DIR = Path('manuscript_figures')
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

ALPHAS = (0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50)
Q0S = (1.0, 1.5, 2.0, 2.5, 3.0)
MOMENT_NAMES = ('Mean', 'Variance', 'Third central moment', 'Fourth central moment', 'Fifth central moment')
SAMPLE_INDEX = 0     # same deterministic file index for all 35 configurations
EXAMPLE_ALPHA = 0.20 # for 1x5 figure
EXAMPLE_Q0 = 1.0
PHI_RANGE = (-0.5, 0.5)
ETA_RANGE = (-0.5, 0.5)
CMAP = 'jet'          # matches original uploaded figure's appearance

if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(f'Dataset root not found: {DATASET_ROOT}')

def config_name(alpha, q0):
    return f'c0005_alpha{alpha:.2f}_q_{q0:.1f}'.replace('.', 'p')

def load_representatives(sample_index=SAMPLE_INDEX):
    samples, paths = {}, {}
    for q0 in Q0S:
        for alpha in ALPHAS:
            folder = DATASET_ROOT / config_name(alpha, q0)
            filenames = sorted(folder.glob('agg500_event_*.npy'))
            if len(filenames) <= sample_index:
                raise FileNotFoundError(f'{folder}: found {len(filenames)} samples, need index {sample_index}')
            path = filenames[sample_index]
            array = np.load(path, allow_pickle=False)
            if array.shape != (5,32,32) or not np.isfinite(array).all():
                raise ValueError(f'Invalid shape or values in {path}: {array.shape}')
            samples[(alpha, q0)] = array
            paths[(alpha, q0)] = path
    return samples, paths

samples, sample_paths = load_representatives()
print(f'Loaded {len(samples)} representative aggregate tensors.')

In [ ]:
# Consistent scale across the 35 panels *within* each moment figure.
# For signed moments use a symmetric zero-centered scale if any selected pixels are negative.
def moment_scale(moment_index):
    values = np.stack([samples[(a,q)][moment_index] for q in Q0S for a in ALPHAS])
    lo, hi = float(values.min()), float(values.max())
    if moment_index in (2,4) and lo < 0:
        bound = max(abs(lo), abs(hi), 1e-12)
        return TwoSlopeNorm(vmin=-bound, vcenter=0, vmax=bound)
    return Normalize(vmin=0, vmax=max(hi,1e-12))

def save_figure(fig, name):
    fig.savefig(OUTPUT_DIR / f'{name}.pdf', bbox_inches='tight', facecolor='white')
    fig.savefig(OUTPUT_DIR / f'{name}.png', bbox_inches='tight', facecolor='white', dpi=300)


def plot_configuration_grid(moment_number=1, save=True):
    """5 rows of Q0 × 7 columns of alpha_s, like the supplied 4x3 figure."""
    k = moment_number - 1
    if k not in range(5):
        raise ValueError('moment_number must be 1..5')
    norm = moment_scale(k)
    fig, axes = plt.subplots(5, 7, figsize=(14.8, 11.6),
                             gridspec_kw={'wspace': 0.12, 'hspace': 0.14},
                             sharex=True, sharey=True)
    for r, q0 in enumerate(Q0S):
        for c, alpha in enumerate(ALPHAS):
            ax = axes[r,c]
            # Stored [phi, eta], transpose to display phi horizontally, eta vertically.
            ax.imshow(samples[(alpha,q0)][k].T, origin='lower', cmap=CMAP,
                      extent=(*PHI_RANGE, *ETA_RANGE), interpolation='nearest',
                      aspect='equal', norm=norm)
            if r == 0:
                ax.set_title(rf'$\alpha_s={alpha:.2f}$', fontsize=11, pad=8)
            if c == 0:
                ax.set_ylabel(rf'$Q_0={q0:.1f}\,\mathrm{{GeV}}$', fontsize=10, labelpad=11)
            if r == len(Q0S)-1:
                ax.set_xticks([-0.5, 0.0, 0.5])
            else:
                ax.set_xticks([])
            # Physical coordinate ticks
            ax.set_xlim(-0.5, 0.5)
            ax.set_ylim(-0.5, 0.5)

            ax.set_xticks([-0.5, 0.0, 0.5])
            ax.set_yticks([-0.5, 0.0, 0.5])

            # Show phi values only on the bottom row
            ax.tick_params(
                axis="x",
                bottom=(r == len(Q0S) - 1),
                labelbottom=(r == len(Q0S) - 1)
            )

            # Show eta values only on the first column
            ax.tick_params(
                axis="y",
                left=(c == 0),
                labelleft=(c == 0)
            )

            ax.tick_params(labelsize=8, length=3, pad=3)
    fig.subplots_adjust(left=0.10, right=0.98, top=0.92, bottom=0.16, 
                        wspace=0.0,  # Horizontal gap between subplots
                        hspace=0.0   # Vertical gap between subplots 
    )
    fig.supxlabel(r'$\phi$ (azimuthal angle)', y=0.103, fontsize=12)
    # fig.supylabel(r'$\eta$ (pseudorapidity)', x=0.015, fontsize=12)
    fig.supylabel(r'$\eta$ (pseudorapidity)', x=0.045, fontsize=12)
    # # Shared horizontal colorbar below all panels (same as reference PDF).
    # cax = fig.add_axes([0.24, 0.050, 0.58, 0.016])
    # cb = fig.colorbar(ScalarMappable(norm=norm, cmap=CMAP), cax=cax,
    #                   orientation='horizontal')
    # cb.set_label(rf'Normalized $M_{{{moment_number}}}$ intensity', fontsize=10)
    # cb.ax.tick_params(labelsize=8)
    # Shared horizontal colorbar
    cax = fig.add_axes([0.30, 0.050, 0.52, 0.016])

    cb = fig.colorbar(
        ScalarMappable(norm=norm, cmap=CMAP),
        cax=cax,
        orientation='horizontal'
    )

    cb.ax.tick_params(labelsize=8)

    if moment_number == 1:
        # Label positioned to the LEFT of the colorbar
        fig.text(
            0.28, 0.058,
            rf'Normalized First-Order $p_T$ Moment ($M_{{{moment_number}}}$)',
            fontsize=11,
            ha='right',
            va='center'
        )
    else:
        fig.text(
            0.28, 0.058,
            rf'Normalized $p_T$ Moment ($M_{{{moment_number}}}$)',
            fontsize=11,
            ha='right',
            va='center'
        )
    
    dataset_note = (
        'Dataset generation parameters\n'
        r'$\sqrt{s_{\mathrm{NN}}}=5.02$ TeV' '\n'
        r'$\hat{p}_T \in [50,250]$ GeV' '\n'
        'Centrality: 0–5%\n'
        r'Selected jet $p_T \in [90,110]$ GeV' '\n'
        r'Jet radius $R=0.4$'
    )

    fig.text(
        0.985, 0.022,     # Lower y value moves the box DOWN
        dataset_note,
        ha='right',
        va='bottom',
        fontsize=9,
        linespacing=1.4,
        bbox=dict(
            boxstyle='round,pad=0.35',
            facecolor='white',
            edgecolor='gray',
            alpha=0.72
        )
    )
    fig.subplots_adjust(
        left=0.10,
        right=0.98,
        top=0.90,
        bottom=0.16,
        wspace=0.04,
        hspace=0.06
    )
    if save:
        save_figure(fig, f'M{moment_number}_grid_5x7')
    plt.show()
    plt.close(fig)

# Main-paper figure: M1 across all 35 parameter configurations
plot_configuration_grid(moment_number=1)

In [ ]:

def plot_configuration_grid(moment_number=1, save=True):
    """
    Compact 5x7 configuration grid.

    Rows: Q0 = 1.0, 1.5, 2.0, 2.5, 3.0 GeV
    Columns: alpha_s = 0.20, ..., 0.50

    Matches the 1x5 moment figure:
      - tightly packed square jet images
      - phi under every subplot
      - eta beside first column
      - no numeric axis ticks
      - vertical colorbar on the right
      - transparent information boxes
    """

    k = moment_number - 1
    if k not in range(5):
        raise ValueError("moment_number must be 1..5")

    norm = moment_scale(k)

    fig, axes = plt.subplots(
        len(Q0S),
        len(ALPHAS),
        figsize=(16, 10.5),
        sharex=True,
        sharey=True
    )

    for r, q0 in enumerate(Q0S):
        for c, alpha in enumerate(ALPHAS):

            ax = axes[r, c]

            # Stored tensor axes are [phi, eta].
            ax.imshow(
                samples[(alpha, q0)][k].T,
                origin='lower',
                extent=(*PHI_RANGE, *ETA_RANGE),
                cmap=CMAP,
                norm=norm,
                interpolation='nearest',
                aspect='equal'
            )

            # Column headings: alpha_s
            if r == 0:
                ax.set_title(
                    rf'$\alpha_s={alpha:.2f}$',
                    fontsize=11,
                    pad=6
                )

            # Physical coordinate limits
            ax.set_xlim(*PHI_RANGE)
            ax.set_ylim(*ETA_RANGE)

            # Remove numerical ticks on every subplot
            ax.set_xticks([])
            ax.set_yticks([])

            # Show phi only at the bottom of each column
            if r == len(Q0S) - 1:
                ax.set_xlabel(
                    r'$\phi$',
                    fontsize=11,
                    labelpad=3
                )

            # Eta beside the first column
            if c == 0:
                ax.set_ylabel(
                    r'$\eta$',
                    fontsize=10,
                    labelpad=1
                )

    # Tight alignment, with room on right for annotations

    fig.subplots_adjust(
        left=0.09,
        right=0.72,
        top=0.94,
        bottom=0.26,   # Reserve space for all three elements
        wspace=0.015,
        hspace=0.015
    )

    # Vertical Q0 labels, positioned close to each row
    for r, q0 in enumerate(Q0S):
        pos = axes[r, 0].get_position()

        fig.text(
            pos.x0 - 0.02,             # Distance from the first subplot
            (pos.y0 + pos.y1) / 2,
            rf'$Q_0={q0:.1f}\,\mathrm{{GeV}}$',
            rotation=90,
            ha='center',
            va='center',
            fontsize=11
        )




    # ============================================================
    # Bottom layout: Colorbar | Angular Coordinates | Dataset
    # ============================================================

    # 1. Horizontal colorbar — bottom left
    cax = fig.add_axes([0.09, 0.17, 0.30, 0.018])

    cb = fig.colorbar(
        ScalarMappable(norm=norm, cmap=CMAP),
        cax=cax,
        orientation='horizontal'
    )

    cb.ax.tick_params(labelsize=8)

    if moment_number == 1:
        colorbar_label = r'Log-Normalized Mean $p_T$ ($M_1$)'
    elif moment_number in (3, 5):
        colorbar_label = (
            rf'Signed-Log-Normalized $p_T$ Moment '
            rf'($M_{{{moment_number}}}$)'
        )
    else:
        colorbar_label = (
            rf'Log-Normalized $p_T$ Moment '
            rf'($M_{{{moment_number}}}$)'
        )

    # Label above the horizontal colorbar
    cb.ax.set_title(
        colorbar_label,
        fontsize=10,
        pad=9
    )


    # 2. Angular coordinates — bottom center
    coordinate_note = (
        'pseudorapidity:' r'$\eta \in [-0.5,\,0.5]$' '\n'
        'azimuthal angle: 'r'$\phi \in [-0.5,\,0.5]$' '\n'
        'transverse momentum:' r'$p_T$'
    )

    fig.text(
        0.42, 0.22,
        coordinate_note,
        ha='left',
        va='top',
        fontsize=9,
        linespacing=1.5,
        bbox=dict(
            boxstyle='round,pad=0.35',
            facecolor='white',
            edgecolor='gray',
            alpha=0.72
        )
    )


    # 3. Dataset generation parameters — bottom right
    dataset_note = (
        'Dataset generation parameters\n'
        r'$\sqrt{s_{\mathrm{NN}}}=5.02$ TeV' '\n'
        r'$\hat{p}_T \in [50,250]$ GeV' '\n'
        'Centrality: 0–5%\n'
        r'Selected jet $p_T \in [90,110]$ GeV' '\n'
        r'Jet radius $R=0.4$'
    )

    fig.text(
        0.58, 0.22,
        dataset_note,
        ha='left',
        va='top',
        fontsize=9,
        linespacing=1.4,
        bbox=dict(
            boxstyle='round,pad=0.35',
            facecolor='white',
            edgecolor='gray',
            alpha=0.72
        )
    )

    if save:
        save_figure(
            fig,
            f'M{moment_number}_grid_5x7'
        )

    plt.show()
    plt.close(fig)


# Main paper: M1 configuration grid
plot_configuration_grid(moment_number=1)


In [ ]:

def plot_configuration_grid(moment_number=1, save=True):
    """
    Compact 7x5 configuration grid.

    Rows: alpha_s = 0.20, 0.25, ..., 0.50
    Columns: Q0 = 1.0, 1.5, 2.0, 2.5, 3.0 GeV

    Layout:
      - tightly packed square jet images
      - phi only below the bottom row
      - eta beside the first column
      - alpha_s values vertically on the left
      - Q0 values above each column
      - no numerical axis ticks
      - bottom colorbar and two annotation boxes
    """

    k = moment_number - 1
    if k not in range(5):
        raise ValueError("moment_number must be 1..5")

    norm = moment_scale(k)

    # 7 rows x 5 columns
    # fig, axes = plt.subplots(
    #     len(ALPHAS),
    #     len(Q0S),
    #     figsize=(12.5, 15),
    #     sharex=True,
    #     sharey=True
    # )
    fig, axes = plt.subplots(
        len(ALPHAS),
        len(Q0S),
        figsize=(10, 15.4),
        sharex=True,
        sharey=True
    )
    for r, alpha in enumerate(ALPHAS):
        for c, q0 in enumerate(Q0S):

            ax = axes[r, c]

            # Stored tensor axes: [phi, eta]
            # Transpose only for display.
            ax.imshow(
                samples[(alpha, q0)][k].T,
                origin='lower',
                extent=(*PHI_RANGE, *ETA_RANGE),
                cmap=CMAP,
                norm=norm,
                interpolation='nearest',
                aspect='equal'
            )

            # Column headings: Q0
            if r == 0:
                ax.set_title(
                    rf'$Q_0={q0:.1f}\,\mathrm{{GeV}}$',
                    fontsize=11,
                    pad=6
                )

            # Physical coordinate limits
            ax.set_xlim(*PHI_RANGE)
            ax.set_ylim(*ETA_RANGE)

            # Remove numerical ticks
            ax.set_xticks([])
            ax.set_yticks([])

            # Phi only below the bottom row
            if r == len(ALPHAS) - 1:
                ax.set_xlabel(
                    r'$\phi$',
                    fontsize=11,
                    labelpad=3
                )

            # Eta beside the first column
            if c == 0:
                ax.set_ylabel(
                    r'$\eta$',
                    fontsize=10,
                    labelpad=1
                )

    # Tightly arranged image grid
    # Reserve bottom space for legend and boxes.
    fig.subplots_adjust(
        left=0.13,
        right=0.96,
        top=0.95,
        bottom=0.19,
        wspace=0.02,
        hspace=0.01
    )

    # Vertical alpha_s labels on the left of each row
    for r, alpha in enumerate(ALPHAS):

        pos = axes[r, 0].get_position()

        fig.text(
            pos.x0 - 0.025,
            (pos.y0 + pos.y1) / 2,
            rf'$\alpha_s={alpha:.2f}$',
            rotation=90,
            ha='center',
            va='center',
            fontsize=11
        )

    # ========================================================
    # Bottom layout:
    # Colorbar | Angular coordinates | Dataset parameters
    # ========================================================

    # 1. Horizontal colorbar
    cax = fig.add_axes([0.13, 0.145, 0.30, 0.014])

    cb = fig.colorbar(
        ScalarMappable(norm=norm, cmap=CMAP),
        cax=cax,
        orientation='horizontal'
    )

    cb.ax.tick_params(labelsize=8)

    if moment_number == 1:
        colorbar_label = (
            r'Log-Normalized Mean $p_T$ ($M_1$)'
        )

    elif moment_number in (3, 5):
        colorbar_label = (
            rf'Signed-Log-Normalized $p_T$ Moment '
            rf'($M_{{{moment_number}}}$)'
        )

    else:
        colorbar_label = (
            rf'Log-Normalized $p_T$ Moment '
            rf'($M_{{{moment_number}}}$)'
        )

    cb.ax.set_title(
        colorbar_label,
        fontsize=10,
        pad=9
    )

    # 2. Coordinate definitions
    coordinate_note = (
        'Pseudorapidity: '
        r'$\eta \in [-0.5,\,0.5]$' '\n'
        'Azimuthal angle: '
        r'$\phi \in [-0.5,\,0.5]$' '\n'
        'Transverse momentum: '
        r'$p_T$'
    )

    fig.text(
        0.75, 0.17,
        coordinate_note,
        ha='left',
        va='top',
        fontsize=9,
        linespacing=1.5,
        bbox=dict(
            boxstyle='round,pad=0.35',
            facecolor='white',
            edgecolor='gray',
            alpha=0.72
        )
    )

    # # 3. Dataset generation parameters
    # dataset_note = (
    #     'Dataset generation parameters\n'
    #     r'$\sqrt{s_{\mathrm{NN}}}=5.02$ TeV' '\n'
    #     r'$\hat{p}_T \in [50,250]$ GeV' '\n'
    #     'Centrality: 0–5%\n'
    #     r'Selected jet $p_T \in [90,110]$ GeV' '\n'
    #     r'Jet radius $R=0.4$'
    # )

    # fig.text(
    #     0.70, 0.17,
    #     dataset_note,
    #     ha='left',
    #     va='top',
    #     fontsize=9,
    #     linespacing=1.4,
    #     bbox=dict(
    #         boxstyle='round,pad=0.35',
    #         facecolor='white',
    #         edgecolor='gray',
    #         alpha=0.72
    #     )
    # )

    # Save figure
    if save:
        save_figure(
            fig,
            f'M{moment_number}_grid_7x5'
        )

    plt.show()
    plt.close(fig)


# Main paper: M1 across all 35 configurations
plot_configuration_grid(moment_number=1)


In [ ]:
# Supplementary figures: one 5x7 grid for each higher moment.
for moment_number in (2,3,4,5):
    plot_configuration_grid(moment_number)

In [ ]:
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

def plot_same_aggregate_five_moments(
    alpha=EXAMPLE_ALPHA,
    q0=EXAMPLE_Q0,
    save=True
):
    """Compact 1x5 panel showing M1-M5 from the same 500-jet aggregate."""

    sample = samples[(alpha, q0)]

    # Shared normalized display scale
    norm = Normalize(vmin=0, vmax=1)

    fig, axes = plt.subplots(
        1, 5,
        figsize=(15, 3.5),
        sharex=True,
        sharey=True
    )

    for k, ax in enumerate(axes):

        im = ax.imshow(
            sample[k].T,
            origin='lower',
            extent=(*PHI_RANGE, *ETA_RANGE),
            cmap=CMAP,
            norm=norm,
            interpolation='nearest',
            aspect='equal'
        )

        # Moment labels
        ax.set_title(
            rf'$M_{{{k+1}}}$: {MOMENT_NAMES[k]}',
            fontsize=11,
            pad=5
        )

        # Physical coordinate range
        ax.set_xlim(*PHI_RANGE)
        ax.set_ylim(*ETA_RANGE)

        # Remove numerical ticks
        ax.set_xticks([])
        ax.set_yticks([])

        # Keep phi under every subplot
        ax.set_xlabel(r'$\phi$', fontsize=11, labelpad=2)

        # Keep eta only on the first subplot
        if k == 0:
            ax.set_ylabel(r'$\eta$', fontsize=11, labelpad=2)

    # Tight alignment of all five jet images
    fig.subplots_adjust(
        left=0.055,
        right=0.84,
        top=0.86,
        bottom=0.18,
        wspace=0.015
    )

    # Vertical colorbar on the right
    # cax = fig.add_axes([0.855, 0.24, 0.014, 0.52])
    cax = fig.add_axes([0.855, 0.41, 0.014, 0.43])
    cb = fig.colorbar(
        ScalarMappable(norm=norm, cmap=CMAP),
        cax=cax,
        orientation='vertical'
    )
    cb.ax.tick_params(labelsize=8)
    cb.set_label(r'Log-Normalized $p_T$', fontsize=10, labelpad=8,loc='center')


    # Coordinate range box
    coordinate_note = (
        
        r'$\eta \in [-0.5,\,0.5]$' '\n'
        r'$\phi \in [-0.5,\,0.5]$'
    )


    fig.text(
        0.855, 0.33,
        coordinate_note,
        ha='left',
        va='top',
        fontsize=9,
        linespacing=1.5,
        bbox=dict(
            boxstyle='round,pad=0.35',
            facecolor='white',
            edgecolor='gray',
            alpha=0.72
        )
    )

    if save:
        save_figure(fig, 'same_aggregate_M1_to_M5')

    plt.show()
    plt.close(fig)


plot_same_aggregate_five_moments()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

COLORMAPS = [
    'jet',
    'inferno',
    'magma',
    'viridis',
    'plasma',
    'cividis'
]

def compare_colormaps(
    alpha=EXAMPLE_ALPHA,
    q0=EXAMPLE_Q0,
    moment_number=1
):
    sample = samples[(alpha, q0)]
    k = moment_number - 1

    fig, axes = plt.subplots(
        2, 3,
        figsize=(11, 7)
    )

    # Same scaling for a fair comparison
    norm = moment_scale(k)

    for ax, cmap in zip(axes.flat, COLORMAPS):
        im = ax.imshow(
            sample[k].T,
            origin='lower',
            extent=(*PHI_RANGE, *ETA_RANGE),
            cmap=cmap,
            norm=norm,
            interpolation='nearest',
            aspect='equal'
        )

        ax.set_title(cmap, fontsize=12)
        ax.set_xticks([])
        ax.set_yticks([])

    fig.subplots_adjust(
        wspace=0.12,
        hspace=0.18
    )

    plt.show()
    plt.close(fig)


compare_colormaps(moment_number=1)

In [ ]:
# Reproducibility: save exactly which tensor was displayed for each configuration.
manifest = OUTPUT_DIR / 'figure_sample_manifest.csv'
with manifest.open('w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['alpha_s','q0_GeV','sample_index','tensor_path'])
    for q0 in Q0S:
        for alpha in ALPHAS:
            writer.writerow([alpha, q0, SAMPLE_INDEX, str(sample_paths[(alpha,q0)].resolve())])
print('Saved PDF + PNG figures and', manifest.resolve())

**Notes for the manuscript:** All images shown are *stored, already log-normalized* tensors; do not apply another `LogNorm`. The global color scale is shared between configurations **within each moment**; scales differ across moment orders. One selected aggregate per parameter pair illustrates appearance, not the average appearance or a statistically estimated trend.